# Gravitational Wave Inference Demonstration

This notebook demonstrates how to classify noisy data from the LIGO and VIRGO observatories into those segments containing gravitational wave (GW) signals and those segments that do not. Our model is a spatiotemporal-graph AI (hybrid dilated convolution neural networks and graph neural networks, see [arXiv:2306.15728](https://arxiv.org/abs/2306.15728)) that has been trained using distributed training ressources on NCSA Delta. 

In [1]:
import multiprocessing
from concurrent.futures import ThreadPoolExecutor, as_completed

from utils_LEGACY import *

## Data preparation 

Our input data consists of gravitational wave strains h(t). h(t) represents the relative change in distance between test masses in the interferometers due to a passing GW wave as a function of time. This strain data can be analyzed to infer properties of the GW sources (e.g. colliding black holes or neutron stars).

The training segments are 1 second long, consisting of either pure noise or noise with a synthetic GW signal injected. The neural network predicts the probability of a GW at each of the 4096 time steps, with ground truth labels being 0 (no GW) or 1, during the 0.5 seconds before the merger, (GW).

###  SNR scheduling

We employ signal-to-noise ratio (SNR) scheduling in our training, progressively introducing the network to waveforms with lower SNRs as training continues.

In [2]:
noise_ranges = [[0, 0.0], [0, 0.3], [0, 0.9],[0.3, 1.5], [1, 2.0]]
plot_waveforms(wf_dataset, noise_ranges)

NameError: name 'plot_waveforms' is not defined

### Whitening

Whitening is necessary since the noise characteristics of GW detectors are not uniform across all frequencie. We use the known noise power spectral density (PSD) of each detector to whiten/normalize the signals.

In [ ]:
noise_ranges = [[0.1, 0.1], [0.1, 0.1],[1000,1000],[1000,1000]]
plot_whitened_waveforms(wf_dataset, noise_ranges)

## Inference

Our inference dataset comprises real data from the LIGO and VIRGO observatories. It consists of three separate sequences, each 4096 seconds long. The data has already been preprocessed and whitened before we load it here. A gravitational wave is present in each sequence at approximately the 1925-second mark.

### Procedure

The neural network processes these segments to predict the likelihood of gravitational wave events at each time step. Significant peaks in these predictions, which indicate a potential gravitational wave signals, are identified using specified width and height parameters. These identified peaks are then used to determine detection triggers, which are the exact time points where the gravitational waves are detected. 

### Model loading and distribution

Here, we set the inference hyperparameters. We load 4 models onto 4 gpus respectively in order to optimize prediction

In [2]:
###GW strain only###
class InferenceConfig:
    batch_size = 32#64
    feb_data_dir = 'projects/bbvf/victoria/WaveNet_data/month_processed_data'  # put GW strain data directory here
    checkpoint_dir = '/projects/begd/victoria/Wavenet_torch-main/checkpoints/BBH_trained_February2020_validated/' #put model checkpoint directory here
    n_channels = 3
    length = None

inference_args = InferenceConfig()

def load_model(checkpoint_name, device):
    checkpoint_path = os.path.join(inference_args.checkpoint_dir, checkpoint_name)
    state_dict = torch.load(checkpoint_path, map_location=torch.device('cpu'))['state_dict']
    new_state_dict = {key[len("model."):] if key.startswith("model.") else key: value for key, value in state_dict.items()} #this is because there is a slight discrepancy in names between the saved and loaded model
    model = full_module()
    model.load_state_dict(new_state_dict)
    model = model.to(device)
    return model

devices = [torch.device(f'cuda:{i}' if torch.cuda.device_count() > i else 'cpu') for i in range(4)]
models=['2channel_attn_val_loss=0.03209.ckpt', '2channel_attn_val_loss=0.05151.ckpt']#put , separated list of model filenames in the model folder here]
model_filenames = [inference_args.checkpoint_dir+models[0],inference_args.checkpoint_dir+models[1]]#,inference_args.checkpoint_dir+models[2],inference_args.checkpoint_dir+models[3]] #assumes 4 models, reduce if fewer are needed
models = [load_model(filename, device) for filename, device in zip(model_filenames, devices)]

trainable_params = sum(p.numel() for p in models[0].parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params:,}")

print('devices: ',devices)
print(model_filenames)

Trainable parameters: 193,905
devices:  [device(type='cuda', index=0), device(type='cuda', index=1), device(type='cpu'), device(type='cpu')]
['/projects/begd/victoria/Wavenet_torch-main/checkpoints/BBH_trained_February2020_validated/2channel_attn_val_loss=0.03209.ckpt', '/projects/begd/victoria/Wavenet_torch-main/checkpoints/BBH_trained_February2020_validated/2channel_attn_val_loss=0.05151.ckpt']


### Prediction

In [3]:
data_dirs = sorted(glob.glob(f"{inference_args.feb_data_dir}/GW*.hdf5"))
dataset_name_to_dir = {os.path.basename(data_dir): data_dir for data_dir in data_dirs}
dataset_triggers = {os.path.basename(data_dir): [] for data_dir in data_dirs}

thresholds=0.9995
widths=1024

'''total_windows = 0
window_length = 4096
stride_0 = 4096
stride_5 = 2047

for file in data_dirs:
    with h5py.File(file, 'r') as f:
        strain_L1 = f['strain_L1'][:]
        strain_H1 = f['strain_H1'][:]
        strain_V1 = f['strain_L1'][:]
        min_length = min(len(strain_L1), len(strain_H1), len(strain_V1))

        # Number of windows for each stride
        windows_0 = ((min_length - window_length) // stride_0) + 1
        windows_5 = ((min_length - window_length) // stride_5) + 1

        total_windows += windows_0 + windows_5


print(f"Total windows (possible triggers): {total_windows}")'''

'total_windows = 0\nwindow_length = 4096\nstride_0 = 4096\nstride_5 = 2047\n\nfor file in data_dirs:\n    with h5py.File(file, \'r\') as f:\n        strain_L1 = f[\'strain_L1\'][:]\n        strain_H1 = f[\'strain_H1\'][:]\n        strain_V1 = f[\'strain_L1\'][:]\n        min_length = min(len(strain_L1), len(strain_H1), len(strain_V1))\n\n        # Number of windows for each stride\n        windows_0 = ((min_length - window_length) // stride_0) + 1\n        windows_5 = ((min_length - window_length) // stride_5) + 1\n\n        total_windows += windows_0 + windows_5\n\n\nprint(f"Total windows (possible triggers): {total_windows}")'

In [4]:
gc.collect()
torch.cuda.empty_cache()

with ThreadPoolExecutor(max_workers=8*4) as executor:
    futures = []
    for threshold in [thresholds]:
        print(f"threshold: {threshold}", flush=True)
        for width in [widths]:
            print('width: ', width, flush=True)
            for data_dir in data_dirs:
                for model in models:
                    futures.append(executor.submit(process_data, data_dir, model, threshold, width, inference_args))

    for future in as_completed(futures):
        dataset_name, triggers, _, _, _ = future.result()
        dataset_triggers[dataset_name].append(triggers)

threshold: 0.9995
width:  1024


In [5]:
common_triggers_info, common_triggers_count = process_triggers(dataset_name_to_dir, dataset_triggers, models, tolerance=0.005)
file_name = "_".join(re.search(r"val_loss=([0-9]*\.?[0-9]+)", f).group(1) for f in model_filenames)

filename = f"common_triggers{common_triggers_count}_threshold{thresholds}_widths{widths}_models"+file_name+".txt"
with open(filename, 'w') as f:
    f.write("Model Filenames:\n")
    for model_filename in model_filenames:
        f.write(f"{model_filename}\n")
        
    f.write(f"\nthreshold:{thresholds}\n")
    f.write(f"width:{widths}\n")
    
    f.write("\nDataset Names:\n")
    for dataset_name in dataset_name_to_dir.keys():
        f.write(f"{dataset_name}\n")
    
    f.write("\nCommon Triggers:\n")
    for dataset_name, common_triggers in common_triggers_info.items():
        f.write(f"{dataset_name}: {len(common_triggers)} triggers\n")
        f.write(f"Triggers: {common_triggers}\n")

print(f"Information written to {filename}")

Information written to common_triggers0_threshold0.9995_widths1024_models0.03209_0.05151.txt


## Prediction (noise)

In [6]:
#parallelized prediction (across gpus and cpus) to speed up prediction for larger noise directory

class InferenceConfig:
    batch_size = 16#64
    feb_data_dir = '/projects/bbvf/victoria/WaveNet_data/month_processed_data/'  # put GW strain data directory here
    checkpoint_dir = '/projects/begd/victoria/Wavenet_torch-main/checkpoints/BBH_trained_February2020_validated/'#put checkpoint directory here
    n_channels = 3
    length = None

inference_args = InferenceConfig()

# Store triggers for post-processing
dataset_triggers = {}

# Screening for valid files
data_dirs = sorted(glob.glob(f"{inference_args.feb_data_dir}/truncated_whitened_strains_*.hdf5"))
valid_files = []
for file in data_dirs:
    with h5py.File(file, 'r') as f:
        if 'strain_H1' in f and 'strain_L1' in f:
            size_H1 = f['strain_H1'].size
            size_L1 = f['strain_L1'].size

            # require both channels to be non-empty (and optionally long enough)
            if size_H1 >= 4096 and size_L1 >= 4096:
                valid_files.append(file)
            else:
                print(
                    f"Skipping {file}: "
                    f"H1 size = {size_H1}, L1 size = {size_L1} (too short/empty)"
                )

print(f"Total valid files: {len(valid_files)}")
data_dirs = valid_files  # Use these valid files

# Create lookup dictionaries as before
dataset_name_to_dir = {os.path.basename(data_dir): data_dir for data_dir in data_dirs}
dataset_triggers = {os.path.basename(data_dir): [] for data_dir in data_dirs}

# Process files in batches
batch_size = 5
total_files = len(data_dirs)

Skipping /projects/bbvf/victoria/WaveNet_data/month_processed_data/truncated_whitened_strains_1264984064.hdf5: H1 size = 0, L1 size = 15777216 (too short/empty)
Skipping /projects/bbvf/victoria/WaveNet_data/month_processed_data/truncated_whitened_strains_1265025024.hdf5: H1 size = 15777216, L1 size = 0 (too short/empty)
Skipping /projects/bbvf/victoria/WaveNet_data/month_processed_data/truncated_whitened_strains_1265160192.hdf5: H1 size = 0, L1 size = 15777216 (too short/empty)
Skipping /projects/bbvf/victoria/WaveNet_data/month_processed_data/truncated_whitened_strains_1265393664.hdf5: H1 size = 0, L1 size = 15777216 (too short/empty)
Skipping /projects/bbvf/victoria/WaveNet_data/month_processed_data/truncated_whitened_strains_1266192384.hdf5: H1 size = 6790592, L1 size = 0 (too short/empty)
Skipping /projects/bbvf/victoria/WaveNet_data/month_processed_data/truncated_whitened_strains_1266806784.hdf5: H1 size = 15777216, L1 size = 0 (too short/empty)
Skipping /projects/bbvf/victoria/Wa

In [7]:
gc.collect()
torch.cuda.empty_cache()
    
for i in range(82*batch_size, total_files, batch_size):
    batch_files = data_dirs[i:i + batch_size]
    print(f"\n Processing batch {i // batch_size + 1}/{(total_files + batch_size - 1) // batch_size} with {len(batch_files)} files...\n", flush=True)
    
    with ThreadPoolExecutor(max_workers=32) as executor:
        futures = []
        for threshold in [thresholds]:
            print(f"threshold: {threshold}", flush=True)
            for width in [widths]:
                print(f"width: {width}", flush=True)
                for data_dir in batch_files:
                    for model in models:
                        futures.append(executor.submit(process_data, data_dir, model, threshold, width, inference_args))
        
        # As each future completes, update the triggers dictionary without printing immediate triggers.
        for future in as_completed(futures):
            dataset_name, triggers, _, _, _ = future.result()
            dataset_triggers[dataset_name].append(triggers)
    
    # After finishing the current batch, compute common triggers (using AND logic)
    common_triggers_info, _ = process_triggers(dataset_name_to_dir, dataset_triggers, models, tolerance=0.005, plot=False)
    
    print(f"\n📊 Common triggers after batch {i // batch_size + 1}:")
    for data_dir in batch_files:
        ds_name = os.path.basename(data_dir)
    
        gps = gps_from_filename(ds_name)
        utc = gps_to_utc_string(gps)
    
        if ds_name in common_triggers_info:
            common = common_triggers_info[ds_name]
            print(f"   {utc}  (GPS {gps}): {len(common)} common triggers -> {common}")
        else:
            print(f"   {utc}  (GPS {gps}): No triggers yet.")

    # 🔻 STRICT PER-BATCH CLEANUP (this is the part you asked for)
    del futures
    del common_triggers_info
    del batch_files
    gc.collect()
    torch.cuda.empty_cache()



 Processing batch 83/87 with 5 files...

threshold: 0.9995
width: 1024


/projects/begd/victoria/Wavenet_torch-main/inference/realdata/inference_demo/utils_LEGACY.py:147: RuntimeWarning: invalid value encountered in divide
  strain[:] /= std


Predicting for truncated_whitened_strains_1266925568.hdf5:   0%|          | 0/241 [00:00<?, ?it/s]

Predicting for truncated_whitened_strains_1266909184.hdf5:   0%|          | 0/241 [00:00<?, ?it/s]

Predicting for truncated_whitened_strains_1266929664.hdf5:   0%|          | 0/241 [00:00<?, ?it/s]

...

Predicting for truncated_whitened_strains_1266970624.hdf5:   0%|          | 0/241 [00:00<?, ?it/s]

Predicting for truncated_whitened_strains_1266982912.hdf5:   0%|          | 0/241 [00:00<?, ?it/s]

Predicting for truncated_whitened_strains_1266978816.hdf5:   0%|          | 0/241 [00:00<?, ?it/s]

Predicting for truncated_whitened_strains_1266974720.hdf5:   0%|          | 0/241 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
gc.collect()
torch.cuda.empty_cache()
    
for i in range(19*batch_size, total_files, batch_size):
    batch_files = data_dirs[i:i + batch_size]
    print(f"\n Processing batch {i // batch_size + 1}/{(total_files + batch_size - 1) // batch_size} with {len(batch_files)} files...\n", flush=True)
    
    with ThreadPoolExecutor(max_workers=32) as executor:
        futures = []
        for threshold in [thresholds]:
            print(f"threshold: {threshold}", flush=True)
            for width in [widths]:
                print(f"width: {width}", flush=True)
                for data_dir in batch_files:
                    for model in models:
                        futures.append(executor.submit(process_data, data_dir, model, threshold, width, inference_args))
        
        # As each future completes, update the triggers dictionary without printing immediate triggers.
        for future in as_completed(futures):
            dataset_name, triggers, _, _, _ = future.result()
            dataset_triggers[dataset_name].append(triggers)
    
    # After finishing the current batch, compute common triggers (using AND logic)
    common_triggers_info, _ = process_triggers(dataset_name_to_dir, dataset_triggers, models, tolerance=0.005, plot=False)
    
    print(f"\n📊 Common triggers after batch {i // batch_size + 1}:")
    for data_dir in batch_files:
        ds_name = os.path.basename(data_dir)
    
        gps = gps_from_filename(ds_name)
        utc = gps_to_utc_string(gps)
    
        if ds_name in common_triggers_info:
            common = common_triggers_info[ds_name]
            print(f"   {utc}  (GPS {gps}): {len(common)} common triggers -> {common}")
        else:
            print(f"   {utc}  (GPS {gps}): No triggers yet.")

    # 🔻 STRICT PER-BATCH CLEANUP (this is the part you asked for)
    del futures
    del common_triggers_info
    del batch_files
    gc.collect()
    torch.cuda.empty_cache()



 Processing batch 20/87 with 5 files...

threshold: 0.9995
width: 1024


Predicting for truncated_whitened_strains_1264791552.hdf5:   0%|          | 0/41 [00:00<?, ?it/s]

Predicting for truncated_whitened_strains_1264799744.hdf5:   0%|          | 0/241 [00:00<?, ?it/s]

Predicting for truncated_whitened_strains_1264803840.hdf5:   0%|          | 0/235 [00:00<?, ?it/s]

Predicting for truncated_whitened_strains_1264795648.hdf5:   0%|          | 0/241 [00:00<?, ?it/s]

...

Predicting for truncated_whitened_strains_1265094656.hdf5:   0%|          | 0/235 [00:00<?, ?it/s]

Predicting for truncated_whitened_strains_1265102848.hdf5:   0%|          | 0/237 [00:00<?, ?it/s]

Predicting for truncated_whitened_strains_1265106944.hdf5:   0%|          | 0/241 [00:00<?, ?it/s]

Predicting for truncated_whitened_strains_1265090560.hdf5:   0%|          | 0/241 [00:00<?, ?it/s]

Predicting for truncated_whitened_strains_1265098752.hdf5:   0%|          | 0/241 [00:00<?, ?it/s]